# Heritage Short — Stage 3: Moving Shots with Wan 2.2 TI2V-5B

Run on Kaggle with **GPU T4 x2** (or P100 / Colab).
1. Upload `storyboard.json` and `outputs/keyframes/` as a Kaggle Dataset (e.g. `heritage-keyframes`).
2. Run all cells below. Draft at 480p first to save GPU quota, then upgrade hero shots to 720p.
3. Download `shots_wan/` and place files into `outputs/shots/wan/shot_NN.mp4` locally.

In [ ]:
!git clone https://github.com/Wan-Video/Wan2.2.git
%cd Wan2.2
!pip install -q -r requirements.txt huggingface_hub
!huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir ./Wan2.2-TI2V-5B

In [ ]:
import json, os, subprocess, time

INPUT_DIR = '/kaggle/input/heritage-keyframes'  # Adjust to your Kaggle dataset name
OUT_DIR = '/kaggle/working/shots_wan'
os.makedirs(OUT_DIR, exist_ok=True)

with open(os.path.join(INPUT_DIR, 'storyboard.json'), 'r', encoding='utf-8') as f:
    sb = json.load(f)

# Draft at 832*480 (~480p) on T4 to preserve GPU hours; upgrade key shots to 1280*704 if quota allows
SIZE = '832*480'
PRIORITY_SHOTS = [1, 2, 6, 7, 8, 9, 11]  # Hero emotional shots first

logs = []
for shot in sb['shots']:
    sid = shot['id']
    if shot.get('is_end_card') or sid not in PRIORITY_SHOTS:
        continue
    img = os.path.join(INPUT_DIR, 'keyframes', f'shot_{sid:02d}.png')
    out_mp4 = os.path.join(OUT_DIR, f'shot_{sid:02d}.mp4')
    if not os.path.exists(img) or os.path.exists(out_mp4):
        continue
    t0 = time.time()
    cmd = [
        'python', 'generate.py',
        '--task', 'ti2v-5B',
        '--size', SIZE,
        '--ckpt_dir', './Wan2.2-TI2V-5B',
        '--offload_model', 'True',
        '--convert_model_dtype',
        '--t5_cpu',
        '--image', img,
        '--prompt', shot['motion_prompt'],
        '--base_seed', str(shot.get('seed', 1000 + sid)),
        '--save_file', out_mp4,
    ]
    print(f'Generating shot {sid:02d}: {shot["motion_prompt"]}')
    res = subprocess.run(cmd)
    elapsed = round(time.time() - t0, 2)
    logs.append({'stage': 'shots', 'model': 'Wan2.2-TI2V-5B', 'shot_id': sid, 'size': SIZE, 'seed': shot.get('seed'), 'latency_seconds': elapsed, 'gpu_minutes': round(elapsed / 60, 2), 'status': 'success' if res.returncode == 0 else 'error'})
    print(f'Shot {sid:02d} finished in {elapsed}s (rc={res.returncode})')

with open('/kaggle/working/wan_runs.jsonl', 'w') as f:
    for row in logs:
        f.write(json.dumps(row) + '\n')
print('Done! Download shots_wan/ and wan_runs.jsonl from /kaggle/working')